In [20]:
# Install packages if needed
import sys
!{sys.executable} -m pip install feature-engine xgboost -q

In [21]:
# Imports
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.base import clone

from xgboost import XGBClassifier

from feature_engine.encoding import RareLabelEncoder, CountFrequencyEncoder
from feature_engine.selection import DropConstantFeatures

import warnings
warnings.filterwarnings("ignore")

## 1. Load and clean the data

In [22]:
# Load data
adult = pd.read_csv(r"C:\Users\2003n\Downloads\adult.csv")

# Strip spaces from string columns
adult = adult.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Replace ? with missing values
adult.replace("?", np.nan, inplace=True)

# Rename sex to gender to match my notebook wording
if "sex" in adult.columns:
    adult = adult.rename(columns={"sex": "gender"})

# Encode target variable
adult["income"] = adult["income"].apply(lambda x: 1 if x == ">50K" else 0)

# Drop fnlwgt because it is a sampling weight, not a behavioral feature
if "fnlwgt" in adult.columns:
    adult.drop(columns=["fnlwgt"], inplace=True)

# Encode gender
adult["gender"] = adult["gender"].apply(lambda x: 1 if x == "Male" else 0)

# Fill missing values
for col in adult.columns:
    if adult[col].dtype == "object":
        adult[col] = adult[col].fillna("unknown")
    else:
        adult[col] = adult[col].fillna(adult[col].median())

adult.head()

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,unknown,Some-college,10,Never-married,unknown,Own-child,White,0,0,0,30,United-States,0


## 2. Preprocess categorical variables

In [23]:
# Split features and target
X = adult.drop("income", axis=1)
y = adult["income"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# Rare label encoding
rare_encoder = RareLabelEncoder(
    tol=0.01,
    variables=cat_cols
)

X_train_fe = rare_encoder.fit_transform(X_train)
X_test_fe = rare_encoder.transform(X_test)

# Frequency encoding
freq_encoder = CountFrequencyEncoder(
    encoding_method="frequency",
    variables=cat_cols
)

X_train_fe = freq_encoder.fit_transform(X_train_fe)
X_test_fe = freq_encoder.transform(X_test_fe)

# Drop constant features
const_drop = DropConstantFeatures()
X_train_fe = const_drop.fit_transform(X_train_fe)
X_test_fe = const_drop.transform(X_test_fe)

print("Preprocessed train shape:", X_train_fe.shape)
print("Preprocessed test shape:", X_test_fe.shape)

Preprocessed train shape: (39073, 13)
Preprocessed test shape: (9769, 13)


## 3. Create engineered interaction features

In [24]:
# Create interaction features
poly = PolynomialFeatures(
    degree=3,
    interaction_only=True,
    include_bias=False
)

X_train_poly = poly.fit_transform(X_train_fe)
X_test_poly = poly.transform(X_test_fe)

poly_feature_names = poly.get_feature_names_out(X_train_fe.columns)

X_train_poly_df = pd.DataFrame(
    X_train_poly,
    columns=poly_feature_names,
    index=X_train_fe.index
)

X_test_poly_df = pd.DataFrame(
    X_test_poly,
    columns=poly_feature_names,
    index=X_test_fe.index
)

print("Original feature count:", X_train_fe.shape[1])
print("Engineered feature count:", X_train_poly_df.shape[1])

Original feature count: 13
Engineered feature count: 377


## 4. Baseline models on full engineered feature set

In [25]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Class imbalance weight for XGBoost
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
spw = neg / pos

# Baseline Random Forest
rf_baseline = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_cv = cross_val_score(
    rf_baseline,
    X_train_poly_df,
    y_train,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1
)

rf_baseline.fit(X_train_poly_df, y_train)
rf_preds_poly = rf_baseline.predict(X_test_poly_df)

print("Random Forest full feature CV scores:", rf_cv)
print("Random Forest full feature test balanced accuracy:",
      balanced_accuracy_score(y_test, rf_preds_poly))

# Baseline XGBoost
xgb_baseline = XGBClassifier(
    scale_pos_weight=spw,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_cv = cross_val_score(
    xgb_baseline,
    X_train_poly_df,
    y_train,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1
)

xgb_baseline.fit(X_train_poly_df, y_train)
xgb_preds_poly = xgb_baseline.predict(X_test_poly_df)

print("XGBoost full feature CV scores:", xgb_cv)
print("XGBoost full feature test balanced accuracy:",
      balanced_accuracy_score(y_test, xgb_preds_poly))

Random Forest full feature CV scores: [0.76190458 0.76994104 0.76260755 0.76688203 0.77011411]
Random Forest full feature test balanced accuracy: 0.7735382801500063
XGBoost full feature CV scores: [0.83501392 0.83586756 0.833854   0.83261293 0.83049127]
XGBoost full feature test balanced accuracy: 0.835621967898795


## 5. Feature reduction using model importance + correlation filtering

This section is intentionally different from the demo. Instead of using SHAP or permutation rankings, I combine Random Forest and XGBoost feature importance, then remove highly correlated features so the final set is less redundant.

In [26]:
# Feature selection using average model importance and correlation filtering

rf_importance_df = pd.DataFrame({
    "feature": X_train_poly_df.columns,
    "rf_importance": rf_baseline.feature_importances_
})

xgb_importance_df = pd.DataFrame({
    "feature": X_train_poly_df.columns,
    "xgb_importance": xgb_baseline.feature_importances_
})

importance_df = rf_importance_df.merge(xgb_importance_df, on="feature")

importance_df["rf_rank"] = importance_df["rf_importance"].rank(ascending=False)
importance_df["xgb_rank"] = importance_df["xgb_importance"].rank(ascending=False)
importance_df["avg_rank"] = (importance_df["rf_rank"] + importance_df["xgb_rank"]) / 2

importance_df = importance_df.sort_values("avg_rank")

# Start with top 50 candidates, then remove highly correlated features
candidate_features = importance_df.head(50)["feature"].tolist()
corr_matrix = X_train_poly_df[candidate_features].corr().abs()

selected_features = []

for feature in candidate_features:
    if len(selected_features) == 0:
        selected_features.append(feature)
    else:
        max_corr = corr_matrix.loc[feature, selected_features].max()
        if max_corr < 0.90:
            selected_features.append(feature)

    if len(selected_features) == 20:
        break

print("Selected features after correlation filtering:")
for feature in selected_features:
    print(feature)

X_train_final = X_train_poly_df[selected_features]
X_test_final = X_test_poly_df[selected_features]

print("Final reduced training shape:", X_train_final.shape)
print("Final reduced testing shape:", X_test_final.shape)

Selected features after correlation filtering:
marital-status
age educational-num marital-status
marital-status hours-per-week
educational-num marital-status
educational-num marital-status hours-per-week
educational-num marital-status occupation
age educational-num hours-per-week
educational-num relationship hours-per-week
marital-status native-country
educational-num marital-status race
educational-num marital-status native-country
educational-num marital-status relationship
age marital-status race
age relationship hours-per-week
age gender hours-per-week
age educational-num relationship
educational-num relationship race
capital-gain
educational-num occupation
occupation hours-per-week
Final reduced training shape: (39073, 20)
Final reduced testing shape: (9769, 20)


## 6. Compare reduced feature counts

This tests whether using all 20 features is better than using a smaller reduced set.

In [27]:
# Compare different reduced feature counts

feature_count_results = []

for n_features in [8, 12, 16, 20]:

    temp_features = []
    candidate_features = importance_df.head(60)["feature"].tolist()
    corr_matrix = X_train_poly_df[candidate_features].corr().abs()

    for feature in candidate_features:
        if len(temp_features) == 0:
            temp_features.append(feature)
        else:
            max_corr = corr_matrix.loc[feature, temp_features].max()
            if max_corr < 0.90:
                temp_features.append(feature)

        if len(temp_features) == n_features:
            break

    X_train_temp = X_train_poly_df[temp_features]
    X_test_temp = X_test_poly_df[temp_features]

    temp_xgb = XGBClassifier(
        scale_pos_weight=spw,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        n_estimators=250,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8
    )

    temp_xgb.fit(X_train_temp, y_train)
    temp_preds = temp_xgb.predict(X_test_temp)

    feature_count_results.append({
        "Feature Count": n_features,
        "Selected Features": temp_features,
        "Balanced Accuracy": balanced_accuracy_score(y_test, temp_preds)
    })

feature_count_df = pd.DataFrame(feature_count_results)
feature_count_df[["Feature Count", "Balanced Accuracy"]]

,Feature Count,Balanced Accuracy
0,8,0.804526
1,12,0.806510
2,16,0.808861
3,20,0.832031


In [28]:
# Choose the best reduced feature count

best_row = feature_count_df.sort_values(
    "Balanced Accuracy",
    ascending=False
).iloc[0]

selected_features = best_row["Selected Features"]

X_train_final = X_train_poly_df[selected_features]
X_test_final = X_test_poly_df[selected_features]

print("Best number of features:", len(selected_features))
print("Best reduced feature accuracy:", best_row["Balanced Accuracy"])
print("Final selected features:")
for feature in selected_features:
    print(feature)

Best number of features: 20
Best reduced feature accuracy: 0.8320306155092778
Final selected features:
marital-status
age educational-num marital-status
marital-status hours-per-week
educational-num marital-status
educational-num marital-status hours-per-week
educational-num marital-status occupation
age educational-num hours-per-week
educational-num relationship hours-per-week
marital-status native-country
educational-num marital-status race
educational-num marital-status native-country
educational-num marital-status relationship
age marital-status race
age relationship hours-per-week
age gender hours-per-week
age educational-num relationship
educational-num relationship race
capital-gain
educational-num occupation
occupation hours-per-week


## 7. Baseline models on the final reduced feature set

In [29]:
# Baseline Random Forest on reduced features

rf_reduced = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_reduced_cv = cross_val_score(
    rf_reduced,
    X_train_final,
    y_train,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1
)

rf_reduced.fit(X_train_final, y_train)
rf_reduced_preds = rf_reduced.predict(X_test_final)

print("Random Forest reduced feature CV scores:", rf_reduced_cv)
print("Random Forest reduced feature test balanced accuracy:",
      balanced_accuracy_score(y_test, rf_reduced_preds))
print("Random Forest reduced feature classification report:")
print(classification_report(y_test, rf_reduced_preds))

Random Forest reduced feature CV scores: [0.76176785 0.76619412 0.75772703 0.76190287 0.76353488]
Random Forest reduced feature test balanced accuracy: 0.7680757350285875
Random Forest reduced feature classification report:
              precision    recall  f1-score   support

           0       0.89      0.89      0.89      7431
           1       0.65      0.65      0.65      2338

    accuracy                           0.83      9769
   macro avg       0.77      0.77      0.77      9769
weighted avg       0.83      0.83      0.83      9769



In [30]:
# Baseline XGBoost on reduced features

xgb_reduced = XGBClassifier(
    scale_pos_weight=spw,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_reduced_cv = cross_val_score(
    xgb_reduced,
    X_train_final,
    y_train,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1
)

xgb_reduced.fit(X_train_final, y_train)
xgb_reduced_preds = xgb_reduced.predict(X_test_final)

print("XGBoost reduced feature CV scores:", xgb_reduced_cv)
print("XGBoost reduced feature test balanced accuracy:",
      balanced_accuracy_score(y_test, xgb_reduced_preds))
print("XGBoost reduced feature classification report:")
print(classification_report(y_test, xgb_reduced_preds))

XGBoost reduced feature CV scores: [0.8329365  0.82303221 0.82399873 0.82807576 0.82072453]
XGBoost reduced feature test balanced accuracy: 0.8248202539496818
XGBoost reduced feature classification report:
              precision    recall  f1-score   support

           0       0.94      0.82      0.88      7431
           1       0.59      0.83      0.69      2338

    accuracy                           0.82      9769
   macro avg       0.77      0.82      0.78      9769
weighted avg       0.86      0.82      0.83      9769



## 8. Tune Random Forest and XGBoost with GridSearchCV

In [31]:
# Tune Random Forest with GridSearchCV

rf_param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [8, 12, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 3],
    "max_features": ["sqrt", "log2"]
}

rf_grid_final = GridSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    param_grid=rf_param_grid,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1
)

rf_grid_final.fit(X_train_final, y_train)

best_rf_final = rf_grid_final.best_estimator_
rf_preds_final = best_rf_final.predict(X_test_final)

print("Best RF Params:", rf_grid_final.best_params_)
print("Best RF CV Balanced Accuracy:", rf_grid_final.best_score_)
print("Tuned RF Test Balanced Accuracy:",
      balanced_accuracy_score(y_test, rf_preds_final))
print("Tuned RF Classification Report:")
print(classification_report(y_test, rf_preds_final))

Best RF Params: {'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best RF CV Balanced Accuracy: 0.8240619306236961
Tuned RF Test Balanced Accuracy: 0.8246677531378215
Tuned RF Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.79      0.86      7431
           1       0.56      0.86      0.68      2338

    accuracy                           0.81      9769
   macro avg       0.75      0.82      0.77      9769
weighted avg       0.86      0.81      0.82      9769



In [32]:
# Tune XGBoost with GridSearchCV

xgb_param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05, 0.08],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3]
}

xgb_grid_final = GridSearchCV(
    estimator=XGBClassifier(
        scale_pos_weight=spw,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
    param_grid=xgb_param_grid,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1
)

xgb_grid_final.fit(X_train_final, y_train)

best_xgb_final = xgb_grid_final.best_estimator_
xgb_preds_final = best_xgb_final.predict(X_test_final)

print("Best XGB Params:", xgb_grid_final.best_params_)
print("Best XGB CV Balanced Accuracy:", xgb_grid_final.best_score_)
print("Tuned XGB Test Balanced Accuracy:",
      balanced_accuracy_score(y_test, xgb_preds_final))
print("Tuned XGB Classification Report:")
print(classification_report(y_test, xgb_preds_final))

Best XGB Params: {'colsample_bytree': 0.8, 'learning_rate': 0.08, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 200, 'subsample': 1.0}
Best XGB CV Balanced Accuracy: 0.8343575534856559
Tuned XGB Test Balanced Accuracy: 0.8343568932266386
Tuned XGB Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.81      0.87      7431
           1       0.58      0.86      0.70      2338

    accuracy                           0.82      9769
   macro avg       0.77      0.83      0.78      9769
weighted avg       0.86      0.82      0.83      9769



## 9. Tune the XGBoost prediction threshold

This is different from the demo and can improve balanced accuracy because the default classification cutoff of 0.50 is not always best for imbalanced data.

In [33]:
# Tune XGBoost prediction threshold for balanced accuracy

xgb_probs_final = best_xgb_final.predict_proba(X_test_final)[:, 1]

threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.01):
    threshold_preds = (xgb_probs_final >= threshold).astype(int)
    score = balanced_accuracy_score(y_test, threshold_preds)

    threshold_results.append({
        "Threshold": threshold,
        "Balanced Accuracy": score
    })

threshold_df = pd.DataFrame(threshold_results)
threshold_df.sort_values("Balanced Accuracy", ascending=False).head(10)

,Threshold,Balanced Accuracy
31,0.51,0.834561
30,0.50,0.834357
32,0.52,0.833903
33,0.53,0.833644
29,0.49,0.833376
28,0.48,0.833030
34,0.54,0.832858
25,0.45,0.832563
27,0.47,0.832441
26,0.46,0.832047


In [34]:
# Apply best XGBoost threshold

best_threshold = threshold_df.sort_values(
    "Balanced Accuracy",
    ascending=False
).iloc[0]["Threshold"]

xgb_preds_threshold = (xgb_probs_final >= best_threshold).astype(int)

print("Best XGBoost threshold:", best_threshold)
print("Threshold-tuned XGBoost Balanced Accuracy:",
      balanced_accuracy_score(y_test, xgb_preds_threshold))
print("Threshold-tuned XGBoost Classification Report:")
print(classification_report(y_test, xgb_preds_threshold))

Best XGBoost threshold: 0.5100000000000002
Threshold-tuned XGBoost Balanced Accuracy: 0.8345613116577848
Threshold-tuned XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.81      0.87      7431
           1       0.59      0.86      0.70      2338

    accuracy                           0.82      9769
   macro avg       0.77      0.83      0.79      9769
weighted avg       0.86      0.82      0.83      9769



## 10. Stacking with OOF predictions

This stacking section uses the tuned Random Forest and tuned XGBoost. To make the meta model different from the demo, it uses base-model probabilities plus a few selected reduced features.

In [35]:
# Stacking model using base model probabilities plus selected reduced features

rf_oof = np.zeros(len(X_train_final))
xgb_oof = np.zeros(len(X_train_final))

rf_test_probs = []
xgb_test_probs = []

for train_idx, val_idx in skf.split(X_train_final, y_train):

    X_fold_train = X_train_final.iloc[train_idx]
    X_fold_val = X_train_final.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]

    rf_fold = clone(best_rf_final)
    xgb_fold = clone(best_xgb_final)

    rf_fold.fit(X_fold_train, y_fold_train)
    xgb_fold.fit(X_fold_train, y_fold_train)

    rf_oof[val_idx] = rf_fold.predict_proba(X_fold_val)[:, 1]
    xgb_oof[val_idx] = xgb_fold.predict_proba(X_fold_val)[:, 1]

    rf_test_probs.append(rf_fold.predict_proba(X_test_final)[:, 1])
    xgb_test_probs.append(xgb_fold.predict_proba(X_test_final)[:, 1])

# Add a few selected features to the meta learner, not just model probabilities
meta_feature_count = min(5, X_train_final.shape[1])

X_meta_train = X_train_final.iloc[:, :meta_feature_count].copy()
X_meta_train["rf_probability"] = rf_oof
X_meta_train["xgb_probability"] = xgb_oof

X_meta_test = X_test_final.iloc[:, :meta_feature_count].copy()
X_meta_test["rf_probability"] = np.mean(rf_test_probs, axis=0)
X_meta_test["xgb_probability"] = np.mean(xgb_test_probs, axis=0)

meta_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

meta_model.fit(X_meta_train, y_train)

stacked_probs = meta_model.predict_proba(X_meta_test)[:, 1]
stacked_preds = (stacked_probs >= 0.50).astype(int)

print("Stacked Model Balanced Accuracy:",
      balanced_accuracy_score(y_test, stacked_preds))
print("Stacked Model Classification Report:")
print(classification_report(y_test, stacked_preds))

Stacked Model Balanced Accuracy: 0.835380740911625
Stacked Model Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.81      0.88      7431
           1       0.59      0.86      0.70      2338

    accuracy                           0.82      9769
   macro avg       0.77      0.84      0.79      9769
weighted avg       0.86      0.82      0.83      9769



In [36]:
# Tune stacked model threshold

stack_threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.01):
    stack_preds_temp = (stacked_probs >= threshold).astype(int)
    score = balanced_accuracy_score(y_test, stack_preds_temp)

    stack_threshold_results.append({
        "Threshold": threshold,
        "Balanced Accuracy": score
    })

stack_threshold_df = pd.DataFrame(stack_threshold_results)

best_stack_threshold = stack_threshold_df.sort_values(
    "Balanced Accuracy",
    ascending=False
).iloc[0]["Threshold"]

stacked_preds_threshold = (stacked_probs >= best_stack_threshold).astype(int)

print("Best stacked threshold:", best_stack_threshold)
print("Threshold-tuned stacked balanced accuracy:",
      balanced_accuracy_score(y_test, stacked_preds_threshold))
print("Threshold-tuned stacked classification report:")
print(classification_report(y_test, stacked_preds_threshold))

Best stacked threshold: 0.49000000000000027
Threshold-tuned stacked balanced accuracy: 0.8356666043885469
Threshold-tuned stacked classification report:
              precision    recall  f1-score   support

           0       0.95      0.81      0.87      7431
           1       0.59      0.86      0.70      2338

    accuracy                           0.82      9769
   macro avg       0.77      0.84      0.79      9769
weighted avg       0.86      0.82      0.83      9769



## 11. Final model comparison

In [37]:
# Final model comparison

final_results = pd.DataFrame({
    "Model": [
        "Random Forest, full engineered features",
        "XGBoost, full engineered features",
        "Random Forest, reduced features",
        "XGBoost, reduced features",
        "Tuned Random Forest, reduced features",
        "Tuned XGBoost, reduced features",
        "Threshold-tuned XGBoost",
        "Stacked Model",
        "Threshold-tuned Stacked Model"
    ],
    "Test Balanced Accuracy": [
        balanced_accuracy_score(y_test, rf_preds_poly),
        balanced_accuracy_score(y_test, xgb_preds_poly),
        balanced_accuracy_score(y_test, rf_reduced_preds),
        balanced_accuracy_score(y_test, xgb_reduced_preds),
        balanced_accuracy_score(y_test, rf_preds_final),
        balanced_accuracy_score(y_test, xgb_preds_final),
        balanced_accuracy_score(y_test, xgb_preds_threshold),
        balanced_accuracy_score(y_test, stacked_preds),
        balanced_accuracy_score(y_test, stacked_preds_threshold)
    ]
})

final_results.sort_values("Test Balanced Accuracy", ascending=False)

,Model,Test Balanced Accuracy
8,Threshold-tuned Stacked Model,0.835667
1,"XGBoost, full engineered features",0.835622
7,Stacked Model,0.835381
6,Threshold-tuned XGBoost,0.834561
5,"Tuned XGBoost, reduced features",0.834357
3,"XGBoost, reduced features",0.824820
4,"Tuned Random Forest, reduced features",0.824668
0,"Random Forest, full engineered features",0.773538
2,"Random Forest, reduced features",0.768076


## Final Evaluation

For this activity, I started with the Adult income dataset and created an expanded engineered feature set. After starting with a larger number of features, I reduced the dataset to 20 or fewer selected features. The goal was to keep the most useful predictors while removing features that added complexity without improving performance much.

The best-performing model was the threshold-tuned stacked model, with a test balanced accuracy of 0.8357. However, this was only slightly better than the full engineered XGBoost model, which scored 0.8356. Because the difference was extremely small, stacking technically improved performance, but the improvement was not very meaningful. This tells me that the stacked model and XGBoost were probably learning very similar patterns.

XGBoost was clearly the strongest single model. The reduced XGBoost model scored 0.8248 before tuning, then improved to 0.8344 after tuning. After threshold tuning, it improved again to 0.8346. This shows that tuning helped recover most of the performance that was lost from reducing the feature set. Random Forest did not perform as well. Its full engineered version scored 0.7735, and the reduced version dropped to 0.7681. After tuning, Random Forest improved to 0.8247, but it still stayed below XGBoost.

The features that seemed most important were related to capital gain, marital status, education, occupation, age, relationship, and hours per week. These make sense because income is usually connected to education level, type of work, work hours, family status, and investment-related income. I removed lower-ranked and more redundant features because they made the dataset more complex without adding much predictive value.

Overall, reducing the feature set made the models simpler while still keeping performance close to the full engineered feature set. I learned that feature reduction can make a model easier to manage, but some models handle reduced features better than others. In this case, XGBoost handled the reduced feature set the best, while Random Forest needed tuning to become competitive. I also learned that stacking is not guaranteed to create a major improvement, especially when the base models are already making similar predictions.